# Chapter 4 — The LLMAgent class

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting the backbone LLM of your agent

These notebooks run on **Ollama by default**, the setup the book teaches.
If you do nothing, nothing changes: `make_llm()` starts a local Ollama
service when one isn't already running.

To use OpenAI or Anthropic instead:

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you off the Ollama path.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama regardless
of these settings.

If you opted in but forgot to export the key, you will be prompted for it.
That is the safe path on hosted kernels, and it keeps the key out of the
saved notebook. Setting `OLLAMA_API_KEY` alone routes Ollama to Ollama
Cloud.

**Caveat:** the examples are tuned for qwen3. Output on gpt-5 or Claude
will differ from what is printed in the book, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Setting up the LLM

I recommend running this notebook on **Ollama Cloud**. It serves much larger models than most laptops can run, and the outputs in this notebook were recorded on `deepseek-v4.1-flash`.

1. Create an API key at https://ollama.com/settings/keys.
2. Export it before launching Jupyter: `export OLLAMA_API_KEY=<your key>`.
3. That's all. When the notebook builds its LLM, it detects the key and uses Ollama Cloud.

**Prefer to run locally?** Install Ollama from https://ollama.com/download and pull the model with `ollama pull qwen3:14b`. If no key is set, the notebook starts the Ollama service for you, so there is no need to run `ollama serve`. Local models are smaller, so your outputs will differ from the ones recorded here.

In [2]:
from llm_agents_from_scratch.notebook_utils import make_llm

## Examples

### Example 1: Instantiating an `LLMAgent`

In [3]:
from llm_agents_from_scratch import LLMAgent

llm = make_llm()
llm_agent = LLMAgent(
    llm=llm,
)

✓ Using Ollama Cloud (deepseek-v4.1-flash)


In [4]:
llm_agent.tools

[]

### Example 2: Demo usage of `add_tool()`

In [5]:
from llm_agents_from_scratch.tools import SimpleFunctionTool

def add_one(x: int) -> int:
    """A dummy tool for adding one to the supplied number."""
    return x + 1
    
tool = SimpleFunctionTool(func=add_one)

In [6]:
llm_agent.add_tool(tool)
llm_agent.tools

### Example 3: The Hailstone LLM Agent

In [7]:
LOGGING_ENABLED = True

In [8]:
import logging
from llm_agents_from_scratch.logger import enable_console_logging

if LOGGING_ENABLED:
    enable_console_logging(logging.INFO)

#### Define the Hailstone tool

This is an adapted version of the Hailstone tool from Chapter 2. Since LLMs have been pretrained on a corpus that includes information on the Hailstone sequence, they may rely on their parametric knowledge to perform the task rather than using the provided tool.

One way to force tool-calling is to obfuscate the function details and omit any mention of the Hailstone sequence. This ensures our demonstration shows the LLM agent actually using tools.

In [9]:
from pydantic import BaseModel
from llm_agents_from_scratch.tools import PydanticFunctionTool


class AlgoParams(BaseModel):
    """Params for next_number."""

    x: int


def next_number(params: AlgoParams) -> int:
    """Generate the next number of the sequence."""
    if params.x % 2 == 0:
        return params.x // 2
    return 3 * params.x + 1


# convert our Python function to a BaseTool
tool = PydanticFunctionTool(next_number)

#### Define our backbone LLM

In [10]:
llm = make_llm()

✓ Using Ollama Cloud (deepseek-v4.1-flash)


#### Define the LLMAgent

In [11]:
from llm_agents_from_scratch import LLMAgent

llm_agent = LLMAgent(
    llm=llm,
    tools=[tool],
)

#### The Hailstone Task

In [12]:
from llm_agents_from_scratch.data_structures import Task

In [13]:
instruction_template = """
You are given a tool, `next_number`, that generates the next number in the
sequence given the current number.

Start with the number x={x}.

<rules>
CALL `next_number` on the current number x
STOP AND WAIT for the result.
REPEAT this step-by-step process until the number 1 is reached.
FINAL RESULT: When you receive the number 1, provide the complete sequence you
observed from start to finish (including the starting number x and ending number
1).
</rules>

<warnings>
NEVER fabricate or simulate tool call results
NEVER make multiple tool calls in one response
STOP and WAIT - ALWAYS wait for the actual tool response before deciding next
steps
</warnings>
""".strip()

#### Running the Task

In [14]:
number = 4
sequence = [4, 2, 1]  # correct Hailstone sequence
task = Task(
    instruction=instruction_template.format(x=number),
)

In [15]:
handler = llm_agent.run(task, max_steps=5)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: You are given a tool, `next_number`, that generates the next number in the
sequence given the current number.

Start with the number ...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: You are given a tool, `next_number`, that generates the next number in the
sequence given the current number.

Start with the numb...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 2
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "17259c76-9a5c-45d5-9550-7f47ba25c61d",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the next tool call: use tool `next_number` with argument x=2, then stop and wait for the actual tool result before deciding the ne...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙

In [16]:
handler.done()

False

In [17]:
# number of sub-steps taken
handler.step_counter

1

#### The TaskResult

Upon successful task execution, the final `TaskResult` object is set as the result for the `TaskHandler` (an `asyncio.Future`).

In [18]:
result = handler.result() if not handler.exception() else str(handler.exception())
print(result)

The tool returned 1, which means we've reached the end of the sequence.

The complete sequence observed from start to finish:
**4 → 2 → 1**


In [19]:
# Alternative execution style (if you don't want/need the handler)
# result = await llm_agent.run(task, max_steps=5)

#### The Rollout

The `rollout` attribute of the `TaskHandler` sheds light on the steps that the `LLMAgent` took to perform its task.

In [20]:
print(handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'You are given a tool, `next_number`, that generates the next number in the
sequence given the current number.

Start with the number x=4.

<rules>
CALL `next_number` on the current number x
STOP AND WAIT for the result.
REPEAT this step-by-step process until the number 1 is reached.
FINAL RESULT: When you receive the number 1, provide the complete sequence you
observed from start to finish (including the starting number x and ending number
1).
</rules>

<warnings>
NEVER fabricate or simulate tool call results
NEVER make multiple tool calls in one response
STOP and WAIT - ALWAYS wait for the actual tool response before deciding next
steps
</warnings>'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "4851a8b8-b42f-4df2-b192-484f6772a13e",
    "tool_name": "next_number",
    "arguments": {
        "x": 4
    }
}.

🔧 tool: {
    "tool_call_id": "4851a8b8-b42f-4df2-b192-484f6772a13e",
    "content": "2",